# Playground 06 — One request, narrated end to end

📖 Primer: [docs/00-concepts.md](../docs/00-concepts.md), section 5

The primer lists the 6 steps a request takes through the finished project. This notebook EXECUTES those steps — **one cell per step**, so you can walk a request through the pipeline at your own pace. Same layers as playground 05, now with the play-by-play.

In [ ]:
import json

# ── the cast (tiny versions of the real files) ──────────────────────

BAGS = {   # the model's data (real: BagStore / the database)
    "chanel-flap-001": {"brand": "Chanel", "model": "Classic Flap", "prices": [9500.0, 9800.0]},
    "lv-neverfull-001": {"brand": "Louis Vuitton", "model": "Neverfull MM", "prices": [1800.0]},
    "hermes-birkin-001": {"brand": "Hermès", "model": "Birkin 30", "prices": [22000.0]},
    "coach-tabby-001": {"brand": "Coach", "model": "Tabby 26", "prices": [450.0]},
}


def store_cheapest(n):                      # MODEL  (real: app/store.py)
    ranked = sorted(BAGS.items(), key=lambda item: item[1]["prices"][-1])
    return ranked[:n]


def bag_response(bag_id, bag):              # VIEW   (real: app/schemas.py)
    return {"id": bag_id,
            "label": f"{bag['brand']} {bag['model']}",
            "current_price": bag["prices"][-1]}

In [ ]:
# STEP 1 — the request arrives at the server
#          (real life: FastAPI in app/main.py is listening)
raw_request = "GET /bags/cheapest?n=2"

method, rest = raw_request.split(" ")
path, query = rest.split("?")
print(f'a request knocks: "{raw_request}"')
print(f"method={method!r}  path={path!r}  query={query!r}")

In [ ]:
# STEP 2 — the server matches the path to a CONTROLLER function and
#          parses the query parameter into a real int
n = int(query.removeprefix("n="))           # "n=2" -> 2
print(f"matched: cheapest_bags(n={n})   (n is now an int, not text)")

In [ ]:
# STEP 3 — the controller calls the MODEL: store.cheapest(n)
results = store_cheapest(n)
print(f"the model went looking through {len(BAGS)} bags...")

In [ ]:
# STEP 4 — the model returns the n cheapest Bag objects (R in CRUD)
for bag_id, bag in results:
    print(f"{bag_id}  (current price ${bag['prices'][-1]})")

In [ ]:
# STEP 5 — the controller passes each Bag through the VIEW:
#          full price history hidden, fields renamed
shaped = [bag_response(bag_id, bag) for bag_id, bag in results]
for item in shaped:
    print(item)

In [ ]:
# STEP 6 — the server turns that into JSON text + status 200 and
#          sends it back over the wire
print("HTTP 200 OK")
print(json.dumps(shaped, indent=2, ensure_ascii=False))

Six steps, four concepts (API, CRUD, table, MVC), one request. Every phase you're about to build is one piece of this pipeline.

## ✏️ Your turn

These exercises reuse the step cells above — edit, then re-run steps 1→6 in order.

1. Change `raw_request` to `"GET /bags/cheapest?n=3"` and re-run all six steps. Which cells printed differently? Which didn't change at all?
2. Set `n` BIGGER than the number of bags (`n=10`). Does it crash or cope? Find the line that decides.
3. Break step 2 on purpose: `raw_request = "GET /bags/cheapest?n=two"`. Read the crash. In real FastAPI this exact mistake returns a **422** error to the caller instead of crashing — Phase 4 will show you.
4. Out loud, no peeking: which file in the REAL `app/` folder plays each role? (Steps 1–2: ____, steps 3–4: ____, step 5: ____.) Answers are in the primer's MVC diagram.